## Add Computed Metrics to CSV

In [1]:
"""
Set up the Python path so that the repo-root package `models.classification` is importable,
then import pandas and the settings constants used to locate the metrics CSV and enumerate
registered score functions.
"""

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from settings import METRICS_CSV, METRIC_REGISTRY, metric_col_name

In [ ]:
"""
Load the raw metrics CSV, normalize legacy column names, compute every column in
`METRIC_REGISTRY`, zero-pad `sample_id` for consistent string IDs, and write the
enriched DataFrame back to `METRICS_CSV`.
"""

df = pd.read_csv(METRICS_CSV)
if "clip_target_similarity" in df.columns:
    df = df.rename(columns={"clip_target_similarity": "clip_edited"})
print(f"Loaded: {METRICS_CSV}  shape={df.shape}")

added_cols = []
for key, opt in METRIC_REGISTRY.items():
    col = metric_col_name(key, opt.fn)
    df[col] = opt.fn(df)
    added_cols.append(col)

df["sample_id"] = df["sample_id"].astype(str).str.zfill(12)
df.to_csv(METRICS_CSV, index=False)

print(f"Saved: {METRICS_CSV}")
print(f"Added/updated columns: {added_cols}")
df[added_cols].describe()